In [ ]:
import numpy as np
import h5py
import pandas as pd
import skimage as ski
import matplotlib.pyplot as plt
from pathlib import Path
from stardist.models import StarDist2D
from stardist.plot import render_label
from csbdeep.utils import normalize
import napari
import subprocess
import tifffile
import tempfile
import sys
import preprocess_for_training as prep

In [ ]:
#load stack of the desired name
Data=Path("../Data") #path to folder with tiff stacks and h5 files you want to use (immediately in folder)

def get_magnification(h5_path):
    """
    Assumes that the images are uf or udf, so the images are essentially square
    Input : path of one h5 file
    Output : magnification of the corresponding image file, in nm/pixel
    """
    with h5py.File(h5_path, 'r') as f:
        attrs = f['data'].attrs
        x_ampli = attrs['Scanner.X_Amplitude'] 
        x_points = attrs['Scanner.X_Points'] 
        calibration = attrs['Scanner.X_Calibration']
        mag = (x_ampli * calibration) / x_points 
    return mag 
    
def load_stack(name): #name like FS_250903_007
    stack_path=next(Data.rglob(f"{name}*_uf.tiff"),None) #find files like FS_250903_007_0-85_uf.tiff 
    if stack_path: #if found
        stack=tifffile.imread(stack_path)
    else:
        print("Stack file not found")
        return

    h5_file_path=next(Data.rglob(f"{name}.h5")) #find files like FS_250903_007.h5
    if h5_file_path: #if found
        mag=get_magnification(h5_file_path)
    else:
        print("Metadata h5 file not found")
        return
    #return stack and magnification
    return stack,mag

In [ ]:
a_key = list(stacks.keys())[1]
ex_img = stacks[a_key]["stack"]
print((ex_img.shape))

In [ ]:
#load stardist model for fluo nuclei
stardist_model = StarDist2D.from_pretrained('2D_versatile_fluo')

In [ ]:
TARGET_SIZE=30 #target size of spots, stardist trained to recognize spots of approx this size
OBJECT_SIZE=1.5 #prophyrins is about 1.5nm
def guess_scale_for_stardist(mag):
    return (TARGET_SIZE * mag) / OBJECT_SIZE

In [ ]:
def try_current_model(frame,scale):
    labels, _ = stardist_model.predict_instances(frame, scale=scale, prob_thresh=0.6, nms_thresh=0.3)
    #show original image and result 
    fig,axes=plt.subplots(1,2)
    axes[0].imshow(frame, cmap="inferno")
    axes[0].set_title("Input image")

    axes[1].imshow(render_label(labels,frame))
    axes[1].set_title(f"Stardist labels with scale {scale}")
    plt.show()
    return labels

In [ ]:
def show_labels(frame,labels):
    fig,axes=plt.subplots(1,2)
    axes[0].imshow(frame, cmap="inferno")
    axes[0].set_title("Input image")

    axes[1].imshow(render_label(labels,frame))
    axes[1].set_title(f"Current labels")
    plt.show()

In [ ]:
def try_scales(frame,initial_guess_scale):
    scale_to_try=initial_guess_scale
    while True:
        labels=try_current_model(frame,scale_to_try) #prints images 
        answer = input("Does scale work? If not input a new scale: ")
        if answer == "y": #if scale works, open napari to modify labels by hand
            return labels
        else : #answer is the new scale to try
            scale_to_try=float(answer) #enter loop again and try new scale       

In [ ]:
def save_img_and_labels(img,labels,name):
    save_path = Data / "Training pool"
    save_path.mkdir(exist_ok=True)  # creates such folder if it doesn't exist

    # save image
    images = save_path / "Images"
    images.mkdir(exist_ok=True)
    tifffile.imwrite(images / f"{name}_img.tif", img)

    # save labels
    lbls_folder = save_path / "Labels"
    lbls_folder.mkdir(exist_ok=True)
    tifffile.imwrite(lbls_folder / f"{name}_labels.tif", labels)
    

In [ ]:
#function to call to napari and use standalone script
def use_napari(img,labels):
    temp_dir = Path(tempfile.gettempdir())
    #make temporary files to hand necessary image/labels to standalone script
    img_tmp = temp_dir / "napari_input_img.tif"
    lbl_tmp = temp_dir / "napari_input_lbl.tif"
    out_tmp = temp_dir / "napari_output_lbl.tif"

    #write down arrays from Jupyter notebook memory to .tif files
    tifffile.imwrite(img_tmp, img)
    tifffile.imwrite(lbl_tmp, labels)

    #find standalone script (respective to current working directory)
    script_path = Path.cwd() / "relabel_with_napari.py"

    #run standalone script as subprocess, handing the temp files as text arguments
    #subprocess is blocking call, jupyter notebook will pause until script is finished
    result = subprocess.run([sys.executable, str(script_path), str(img_tmp), str(lbl_tmp), str(out_tmp)],capture_output=True,
    text=True) 
    
    #if script throws error, notebook will print it too
    if result.returncode != 0:
        print("--- STACK TRACE FROM SCRIPT ---")
        print(result.stderr)
        result.check_returncode()  # Raises the error after printing
    

    updated_labels = tifffile.imread(out_tmp)
    return updated_labels

In [ ]:
def modify_labels(frame,labels,save_name):
    while True:
        #show labels and ask if they are good enough
        show_labels(frame,labels)
        answer = input("Are labels ok? [y/n]: ").strip().lower()
        
        #if labels are satisfactory, save to computer
        if answer == "y":
            save_img_and_labels(frame,labels,save_name)
            break #exit loop and continue to next line 
        else: 
            print("Opening napari to modify labels")
            labels=use_napari(frame,labels)
            #repeats loop and asks if these labels are ok          

In [ ]:
#use preprocess to trim stack consistently, return three frames from stack to populate training pool
#choose the target size of spots, use the same for all images so model sees consistent sizes

#choose a stack by name, pre-process it, get original stardist labels, improve with napari, save in training pool
def label_stack(name):

    stack,mag=load_stack(name)
    processed_selection,offsets=prep.apply_to_stack(stack)

    #work on each of the selected frames of this stack
    for i,frame in enumerate(processed_selection):
        #once well labeled, will be saved to computer as
        save_name = name+f'_{i}'
        
        #possibility to skip this frame 
        skip=input("Skip this frame? [y/N]")
        if skip=="y":
           continue 
        
        #visualize original stardist labels with guess for scale, 
        first_guess=guess_scale_for_stardist(mag)
        labels=try_scales(frame,first_guess)

        #open these labels in napari and modify by hand until decide to save to computer, returns nothing 
        modify_labels(frame,labels,save_name)
        print("Moving on to next selected frame of this stack")        